# Day 59 — Multi-Agent Patterns
### Supervisor/worker, parallel agents, self-correction loops, CrewAI overview

## 1. Setup

In [1]:
import anthropic
import json
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated, Literal
import operator

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"
llm = ChatAnthropic(model="claude-haiku-4-5", max_tokens=500)

print("Setup complete — LangGraph + Anthropic ready")

Setup complete — LangGraph + Anthropic ready


## 2. Pattern 1 — Supervisor / Worker

In [2]:
# worker tools
@tool
def calculate(expression: str) -> str:
    """Perform mathematical calculations."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@tool
def analyse_text(text: str) -> str:
    """Analyse text and return word count, sentence count, and average word length."""
    words = text.split()
    sentences = text.count('.') + text.count('!') + text.count('?')
    avg_word_len = round(sum(len(w) for w in words) / len(words), 1) if words else 0
    return json.dumps({
        "word_count": len(words),
        "sentence_count": max(sentences, 1),
        "avg_word_length": avg_word_len
    })

@tool
def summarise(text: str) -> str:
    """Summarise a given text in one sentence."""
    response = client.messages.create(
        model=MODEL, max_tokens=100,
        messages=[{"role": "user", "content": f"Summarise in one sentence: {text}"}]
    )
    return response.content[0].text

# supervisor decides which worker to use
def supervisor(task: str) -> str:
    prompt = f"""You are a task router. Given a task, respond with ONLY one of these words:
- CALCULATE (if the task involves math or numbers)
- ANALYSE (if the task involves text analysis or counting)
- SUMMARISE (if the task involves summarisation)
- ANSWER (if you can answer directly without a tool)

Task: {task}
Response (one word only):"""

    response = client.messages.create(
        model=MODEL, max_tokens=10,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip().upper()

# run the supervisor/worker pipeline
def run_supervisor_worker(task: str):
    print(f"TASK: {task}")
    decision = supervisor(task)
    print(f"SUPERVISOR -> routes to: {decision}")

    if "CALCULATE" in decision:
        expression = task.replace("What is", "").replace("Calculate", "").replace("?", "").strip()
        result = calculate.invoke({"expression": expression})
    elif "ANALYSE" in decision:
        result = analyse_text.invoke({"text": task})
    elif "SUMMARISE" in decision:
        result = summarise.invoke({"text": task})
    else:
        result = client.messages.create(
            model=MODEL, max_tokens=200,
            messages=[{"role": "user", "content": task}]
        ).content[0].text

    print(f"RESULT: {result}\n")
    return result

tasks = [
    "What is 256 * 48?",
    "Analyse this text: The quick brown fox jumps over the lazy dog. It was a sunny day.",
    "Summarise this: Machine learning is a subset of AI that enables systems to learn from data without explicit programming."
]

for task in tasks:
    run_supervisor_worker(task)

TASK: What is 256 * 48?
SUPERVISOR -> routes to: CALCULATE
RESULT: 12288

TASK: Analyse this text: The quick brown fox jumps over the lazy dog. It was a sunny day.
SUPERVISOR -> routes to: ANALYSE
RESULT: {"word_count": 17, "sentence_count": 2, "avg_word_length": 3.9}

TASK: Summarise this: Machine learning is a subset of AI that enables systems to learn from data without explicit programming.
SUPERVISOR -> routes to: SUMMARISE
RESULT: Machine learning allows AI systems to learn and improve automatically from data rather than requiring manual programming.



## 3. Pattern 2 — Parallel Agents

In [4]:
import threading

def run_agent_task(task, results, key):
    response = client.messages.create(
        model=MODEL, max_tokens=200,
        messages=[{"role": "user", "content": task}]
    )
    results[key] = response.content[0].text

def run_parallel_agents(main_task):
    print(f"MAIN TASK: {main_task}\n")

    # split into parallel subtasks
    subtasks = {
        "benefits": f"List 3 key benefits of {main_task} in bullet points. Be concise.",
        "challenges": f"List 3 key challenges of {main_task} in bullet points. Be concise.",
        "use_cases": f"List 3 real-world use cases of {main_task} in bullet points. Be concise."
    }

    results = {}
    threads = []

    # launch all agents simultaneously
    for key, task in subtasks.items():
        t = threading.Thread(target=run_agent_task, args=(task, results, key))
        threads.append(t)
        t.start()

    # wait for all to finish
    for t in threads:
        t.join()

    print("BENEFITS:")
    print(results["benefits"])
    print("\nCHALLENGES:")
    print(results["challenges"])
    print("\nUSE CASES:")
    print(results["use_cases"])
    return results

run_parallel_agents("RAG (Retrieval Augmented Generation)")

MAIN TASK: RAG (Retrieval Augmented Generation)

BENEFITS:
# Key Benefits of RAG

• **Reduces hallucinations** – Grounds responses in actual retrieved documents, minimizing made-up information

• **Uses current knowledge** – Accesses up-to-date external data without retraining the model

• **Improves accuracy** – Provides factual context and citations, making answers more reliable and verifiable

CHALLENGES:
# Key Challenges of RAG

• **Retrieval Quality** - Irrelevant or incomplete documents retrieved can introduce noise and mislead the model, degrading answer accuracy

• **Context Window Limitations** - Large retrieved documents may exceed model context limits, requiring truncation and potentially losing critical information

• **Hallucination & Fact Conflicts** - Models may still generate false information or contradict retrieved documents, especially when sources are ambiguous or conflicting

USE CASES:
# Real-World RAG Use Cases

• **Customer Support Chatbots** - Retrieve relevant

{'benefits': '# Key Benefits of RAG\n\n• **Reduces hallucinations** – Grounds responses in actual retrieved documents, minimizing made-up information\n\n• **Uses current knowledge** – Accesses up-to-date external data without retraining the model\n\n• **Improves accuracy** – Provides factual context and citations, making answers more reliable and verifiable',
 'challenges': '# Key Challenges of RAG\n\n• **Retrieval Quality** - Irrelevant or incomplete documents retrieved can introduce noise and mislead the model, degrading answer accuracy\n\n• **Context Window Limitations** - Large retrieved documents may exceed model context limits, requiring truncation and potentially losing critical information\n\n• **Hallucination & Fact Conflicts** - Models may still generate false information or contradict retrieved documents, especially when sources are ambiguous or conflicting',
 'use_cases': '# Real-World RAG Use Cases\n\n• **Customer Support Chatbots** - Retrieve relevant company documentatio

## 4. Pattern 3 — Self-Correction Loop

In [5]:
def quality_check(answer: str, criteria: str) -> dict:
    prompt = f"""Evaluate if this answer meets the criteria. Respond with ONLY valid JSON.

Criteria: {criteria}
Answer: {answer}

Respond with exactly this JSON format:
{{"passes": true or false, "reason": "brief explanation", "suggestion": "how to improve if failing"}}"""

    response = client.messages.create(
        model=MODEL, max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    )
    text = response.content[0].text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())

def self_correcting_agent(task: str, criteria: str, max_attempts: int = 3):
    print(f"TASK: {task}")
    print(f"CRITERIA: {criteria}\n")

    for attempt in range(1, max_attempts + 1):
        print(f"Attempt {attempt}:")

        response = client.messages.create(
            model=MODEL, max_tokens=300,
            messages=[{"role": "user", "content": task}]
        )
        answer = response.content[0].text
        print(f"  Answer: {answer[:100]}...")

        check = quality_check(answer, criteria)
        print(f"  QC Pass: {check['passes']} | Reason: {check['reason']}")

        if check["passes"]:
            print(f"\nFINAL ANSWER (passed on attempt {attempt}):")
            print(answer)
            return answer
        else:
            print(f"  Retrying... Suggestion: {check['suggestion']}")
            task = f"{task}\n\nIMPORTANT: {check['suggestion']}"

    print("\nMax attempts reached — returning best answer")
    return answer

self_correcting_agent(
    task="Explain what a neural network is.",
    criteria="Must be under 50 words and must include a real-world analogy"
)

TASK: Explain what a neural network is.
CRITERIA: Must be under 50 words and must include a real-world analogy

Attempt 1:
  Answer: # Neural Networks Explained

A neural network is a machine learning model inspired by how brains wor...
  QC Pass: False | Reason: Answer exceeds 50 words significantly (approximately 230 words) and lacks a clear real-world analogy in the main explanation, though a brief one appears at the end.
  Retrying... Suggestion: Condense to under 50 words and integrate a stronger real-world analogy early. Example: 'Neural networks are like a brain learning to recognize faces. They process information through layers of connected nodes (neurons), adjusting connection strengths (weights) based on errors until they accurately predict outcomes—similar to how you improve at a skill through practice and feedback.'
Attempt 2:
  Answer: # Neural Networks

Neural networks are like a brain learning to recognize patterns. They consist of ...
  QC Pass: True | Reason: The answ

'# Neural Networks\n\nNeural networks are like a brain learning to recognize patterns. They consist of interconnected layers of nodes that process information, adjusting connection strengths based on errors until they accurately predict outcomes—similar to how you improve a skill through practice and feedback.'

## 5. CrewAI Overview — The Alternative Framework

In [6]:
# CrewAI comparison -- conceptual demo (install would add heavy deps)
# showing the structural difference vs LangGraph

crewai_equivalent = """
# What the same supervisor/worker pattern looks like in CrewAI:

from crewai import Agent, Task, Crew

# define agents with roles and goals
math_agent = Agent(
    role='Math Specialist',
    goal='Solve mathematical problems accurately',
    backstory='Expert mathematician with years of experience',
    tools=[calculate]
)

text_agent = Agent(
    role='Text Analyst',
    goal='Analyse and summarise text content',
    backstory='Linguistics expert specialising in text analysis',
    tools=[analyse_text, summarise]
)

supervisor_agent = Agent(
    role='Task Supervisor',
    goal='Route tasks to the right specialist and compile results',
    backstory='Experienced project manager who coordinates specialists'
)

# define tasks
task1 = Task(description='Calculate 256 * 48', agent=math_agent)
task2 = Task(description='Analyse the given text', agent=text_agent)
final = Task(description='Compile and present the results', agent=supervisor_agent)

# run the crew
crew = Crew(agents=[math_agent, text_agent, supervisor_agent],
            tasks=[task1, task2, final])
result = crew.kickoff()
"""

print("CrewAI structural pattern:")
print(crewai_equivalent)

print("\nLangGraph vs CrewAI comparison:")
comparison = {
    "LangGraph": {
        "style": "Explicit graph -- you define every node and edge",
        "control": "Full control over state, routing, and execution",
        "best_for": "Complex agents needing precise control and debugging",
        "learning_curve": "Higher -- need to understand graph concepts"
    },
    "CrewAI": {
        "style": "Role-based -- agents have backstories, goals, and personas",
        "control": "Less control -- framework handles routing automatically",
        "best_for": "Quick multi-agent prototypes with clear role separation",
        "learning_curve": "Lower -- intuitive role/task abstraction"
    }
}

for framework, props in comparison.items():
    print(f"\n{framework}:")
    for k, v in props.items():
        print(f"  {k}: {v}")

CrewAI structural pattern:

# What the same supervisor/worker pattern looks like in CrewAI:

from crewai import Agent, Task, Crew

# define agents with roles and goals
math_agent = Agent(
    role='Math Specialist',
    goal='Solve mathematical problems accurately',
    backstory='Expert mathematician with years of experience',
    tools=[calculate]
)

text_agent = Agent(
    role='Text Analyst',
    goal='Analyse and summarise text content',
    backstory='Linguistics expert specialising in text analysis',
    tools=[analyse_text, summarise]
)

supervisor_agent = Agent(
    role='Task Supervisor',
    goal='Route tasks to the right specialist and compile results',
    backstory='Experienced project manager who coordinates specialists'
)

# define tasks
task1 = Task(description='Calculate 256 * 48', agent=math_agent)
task2 = Task(description='Analyse the given text', agent=text_agent)
final = Task(description='Compile and present the results', agent=supervisor_agent)

# run the crew
cr

## 6. Summary — What We Built Today

| Pattern | What it does | When to use it |
|---------|-------------|----------------|
| Supervisor/Worker | Supervisor routes tasks to specialist workers | Different task types need different tools/expertise |
| Parallel Agents | Multiple agents run simultaneously on independent subtasks | Tasks that don't depend on each other — 3x faster than sequential |
| Self-Correction Loop | Agent checks its own output against criteria, retries if failing | Quality-critical tasks where first attempt may not be good enough |
| CrewAI | Role-based multi-agent framework | Quick prototypes with clear role separation |

**Key insight from today:**
The three patterns solve different problems:
- Supervisor/Worker solves ROUTING (which agent should handle this?)
- Parallel agents solve SPEED (how do we do multiple things at once?)
- Self-correction solves QUALITY (how do we ensure the output is good enough?)

A production multi-agent system typically combines all three.